# Lesson 4.6: Time Series in Pandas

**Analyst:** Meron Welderufael
**Date:** 11/06/2026
**Client:** FinFlow (Product Analytics engagement led by Hannah Whitford)
**Notebook purpose:** Produce the cohort churn breakdown Hannah asked for
this morning, plus supporting monthly-signup trend and YoY comparison.
Deliverable for end-of-week.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

df = pd.read_csv('finflow-enriched-200k.csv',
                parse_dates = ['signup_date', 'last_login_date', 'churn_date'])

print(f'Loaded {len(df):,} rows')
print(f'Signup date range: {df['signup_date'].min()} to {df['signup_date'].max()}')
print(f'Columns: {df.shape[1]}')

Loaded 200,000 rows
Signup date range: 2024-10-22 00:00:00 to 2026-10-19 00:00:00
Columns: 22


## Building monthly cohorts

Each customer's signup_date becomes a signup_cohort (their signup month).
This is the cohort identifier we'll group by.

In [2]:
df['signup_cohort'] = df['signup_date'].dt.to_period('M')
#Inspect
print(df[['customer_id', 'signup_date', 'signup_cohort']].head())
print()
print(f'Type of signup_cohort: {df['signup_cohort'].dtype}')
print(f'Number of distinict cohorts: {df['signup_cohort'].nunique()}')

  customer_id signup_date signup_cohort
0    FF-00001  2026-03-11       2026-03
1    FF-00002  2025-12-06       2025-12
2    FF-00003  2026-04-04       2026-04
3    FF-00004  2025-10-25       2025-10
4    FF-00005  2026-02-25       2026-02

Type of signup_cohort: period[M]
Number of distinict cohorts: 25


In [3]:
def churn_rate_pct(active_series):
    return (~active_series).mean() * 100

cohort_churn = df.groupby('signup_cohort').agg(
    customer_count=('customer_id', 'count'),
    churn_rate_pct = ('active', churn_rate_pct),
    avg_monthly_fee = ('monthly_fee', 'mean')   
).round(2)
print(cohort_churn.head(10))
print('---rows omitted---')
print(cohort_churn.tail(5))

               customer_count  churn_rate_pct  avg_monthly_fee
signup_cohort                                                 
2024-10                    60           11.67            41.86
2024-11                   774           19.77            37.27
2024-12                  1779           19.00            33.37
2025-01                  2645           18.83            35.35
2025-02                  3300           19.73            35.81
2025-03                  4457           21.05            35.25
2025-04                  5178           20.01            35.42
2025-05                  6359           20.49            35.29
2025-06                  7145           20.20            34.87
2025-07                  8195           19.77            34.72
---rows omitted---
               customer_count  churn_rate_pct  avg_monthly_fee
signup_cohort                                                 
2026-06                 11794           20.21            35.07
2026-07                  9111       

**Observation:** Cohort churn is flat, 18.9% to 20.7%, with no relationship
to cohort age. Reporting this as "churn is stable across cohorts" would be
technically true and misleading, because current churn rate is the wrong
measure: it mixes cohorts that have had two years to churn with ones that
have had ten days. The honest answer to Hannah is that this cut does not
separate the cohorts, and the measure that would is churn at equal age,
separate the cohorts, and the measure that would is churn at equal age,
e.g. churn rate at six months since signup. Flag that for Diego. That's a
follow-up, not today's deliverable.

In [4]:
cohort_churn.to_csv('cohort-churn-by-signup-month.csv')
print('Saved cohort-churn-by-signup-month.csv')

Saved cohort-churn-by-signup-month.csv


## Monthly signup trend with resample

In [5]:
# Set signup_date as index for resampling
df_indexed = df.set_index('signup_date')

# Monthly signup counts
monthly_signups=df_indexed.resample('ME').size()
print(monthly_signups.head(10))
print(f'Total months: {len(monthly_signups)}')

signup_date
2024-10-31      60
2024-11-30     774
2024-12-31    1779
2025-01-31    2645
2025-02-28    3300
2025-03-31    4457
2025-04-30    5178
2025-05-31    6359
2025-06-30    7145
2025-07-31    8195
Freq: ME, dtype: int64
Total months: 25


In [6]:
# Three-month rolling average of monthly signups
monthly_signups_smoothed = monthly_signups.rolling(window=3).mean()

#compare side by side
comparison = pd.DataFrame({
    'raw': monthly_signups,
    'smoothed 3 month': monthly_signups_smoothed.round(0)
})
print(comparison.head(8))

              raw  smoothed 3 month
signup_date                        
2024-10-31     60               NaN
2024-11-30    774               NaN
2024-12-31   1779            871.00
2025-01-31   2645           1733.00
2025-02-28   3300           2575.00
2025-03-31   4457           3467.00
2025-04-30   5178           4312.00
2025-05-31   6359           5331.00


In [7]:
# Month-over-month percent change in signups
mom_change = monthly_signups.pct_change()*100
print(mom_change.head(8).round(2))

signup_date
2024-10-31       NaN
2024-11-30   1190.00
2024-12-31    129.84
2025-01-31     48.68
2025-02-28     24.76
2025-03-31     35.06
2025-04-30     16.18
2025-05-31     22.81
Freq: ME, dtype: float64


In [8]:
# YoY change in signups
monthly_yoy_prev = monthly_signups.shift(12)
yoy_change = (monthly_signups - monthly_yoy_prev)
yoy_pct = (monthly_signups/monthly_yoy_prev - 1) * 100

yoy_comparison = pd.DataFrame({
    'current' : monthly_signups,
    'prev_year': monthly_yoy_prev,
    'change': yoy_change,
    'pct_change': yoy_pct.round(2)
})
# Show rows where YoY is computable (i.e., second year onward)
print(yoy_comparison.dropna().head(8))

             current  prev_year   change  pct_change
signup_date                                         
2025-10-31     11213      60.00 11153.00    18588.33
2025-11-30     11651     774.00 10877.00     1405.30
2025-12-31     12901    1779.00 11122.00      625.18
2026-01-31     13996    2645.00 11351.00      429.15
2026-02-28     13553    3300.00 10253.00      310.70
2026-03-31     15950    4457.00 11493.00      257.86
2026-04-30     16065    5178.00 10887.00      210.25
2026-05-31     14826    6359.00  8467.00      133.15


In [10]:
#12-month rolling sum of monthly signups(TTM)
ttm_signups = monthly_signups.rolling(window=12).sum()
print(ttm_signups.dropna().head(5))

signup_date
2025-09-30    59104.00
2025-10-31    70257.00
2025-11-30    81134.00
2025-12-31    92256.00
2026-01-31   103607.00
Freq: ME, dtype: float64
